# Qwen video-only α=1 + 30% DivPrune on MELD

Uses the same `benchmark-code` and `meld-dataset` Kaggle inputs as the saved benchmark notebooks. Enable GPU and Internet before running.

Extract video + emotion-task prompt embeddings with 30% visual-token retention, then evaluate the existing full-baseline MLP and existing 30%-DivPrune MLP. Only dev/test are extracted; neither classifier is retrained. The original α=1 run used no fusion normalization, so these video-only embeddings are passed directly to each checkpoint's evaluator.

The first cell contains the checkpoint paths. Include the existing 30%-DivPrune checkpoint in the attached code dataset, or point `DIVPRUNE_CHECKPOINT` to its attached Kaggle input. Its original `070` folder name is misleading: use the checkpoint from `divprune_exp_0.3`.

In [ ]:
from pathlib import Path

CODE = Path("/kaggle/input/datasets/pushkarsingh2005/benchmark-code")
MELD_ROOT = Path("/kaggle/input/datasets/pushkarsingh2005/meld-dataset/MELD.Raw")
BASELINE_CHECKPOINT = CODE / "outputs/qwen2_5_vl_3b_shared/best_mlp.pt"
DIVPRUNE_CHECKPOINT = CODE / "outputs/qwen2_5_vl_3b_divprune_070/new_mlp/best_mlp.pt"
# The checkpoint extracted during the preceding alpha=1 evaluation is also saved here.
if not DIVPRUNE_CHECKPOINT.exists():
    DIVPRUNE_CHECKPOINT = CODE / "outputs/alpha_1_divprune_030_mlp/inputs/outputs/qwen2_5_vl_3b_divprune_070/new_mlp/best_mlp.pt"

INDEX_ROOT = Path("/kaggle/working/outputs/indexes")
CHUNKS_ROOT = Path("/kaggle/working/embeddings/qwen2_5_vl_3b_video_only_divprune_030_chunks")
EMBEDDINGS_ROOT = Path("/kaggle/working/embeddings/qwen2_5_vl_3b_video_only_divprune_030")
RESULTS_ROOT = Path("/kaggle/working/outputs/qwen2_5_vl_3b_video_only_divprune_030")
BASELINE_RESULTS = RESULTS_ROOT / "baseline_mlp"
DIVPRUNE_RESULTS = RESULTS_ROOT / "divprune_030_mlp"
RETAIN_RATIO = 0.30

assert CODE.exists(), f"Missing code dataset: {CODE}"
assert MELD_ROOT.exists(), f"Missing MELD dataset: {MELD_ROOT}"
assert BASELINE_CHECKPOINT.exists(), f"Missing baseline checkpoint: {BASELINE_CHECKPOINT}"
assert DIVPRUNE_CHECKPOINT.exists(), "Set DIVPRUNE_CHECKPOINT to the existing 30%-DivPrune best_mlp.pt in your attached input."
RESULTS_ROOT.mkdir(parents=True, exist_ok=True)
print("Baseline MLP:", BASELINE_CHECKPOINT)
print("30% DivPrune MLP:", DIVPRUNE_CHECKPOINT)


In [ ]:
# Install packages first.
!pip install -q -r {CODE}/requirements.txt


In [ ]:
import json
import pandas as pd
import torch

print("CUDA available:", torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError("Enable a Kaggle GPU accelerator before extraction.")
for i in range(torch.cuda.device_count()):
    print("GPU:", i, torch.cuda.get_device_name(i))

LABELS = ["anger", "disgust", "fear", "joy", "neutral", "sadness", "surprise"]
for checkpoint in [BASELINE_CHECKPOINT, DIVPRUNE_CHECKPOINT]:
    saved = torch.load(checkpoint, map_location="cpu", weights_only=False)
    assert saved["input_dim"] == 2048
    assert saved["label_names"] == LABELS
    print(checkpoint, "best dev macro F1:", saved.get("best_dev_macro_f1"))
del saved


In [ ]:
# Prepare dev/test indexes using the attached MELD videos.
for split in ["dev", "test"]:
    !python {CODE}/scripts/prepare_meld.py \
      --meld-root {MELD_ROOT} \
      --split {split} \
      --output-dir {INDEX_ROOT}

for split in ["dev", "test"]:
    index = pd.read_csv(INDEX_ROOT / f"meld_{split}_index.csv")
    assert len(index) > 0
    assert index["video_path"].map(lambda p: Path(p).is_file()).all()
    print(split, "videos:", len(index))


## Extract video-only embeddings with 30% DivPrune

The utterance is removed from the Qwen input; the emotion-task prompt is retained. Each split uses resumable chunks, matching the shared benchmark workflow.

In [ ]:
# Extract dev: video only + emotion prompt, retain 30% of visual tokens.
!python {CODE}/scripts/run_qwen_shared_chunks.py \
  --index-csv {INDEX_ROOT}/meld_dev_index.csv \
  --chunks-dir {CHUNKS_ROOT}/dev \
  --chunk-prefix dev \
  --chunk-size 500 \
  --fps 6 \
  --max-frames 64 \
  --min-frames 4 \
  --frame-size 224 \
  --pooling last \
  --prompt-style emotion_task \
  --modality-mode video_only \
  --visual-token-pruning divprune \
  --divprune-retain-ratio {RETAIN_RATIO} \
  --save-dtype float32 \
  --gc-every 5 \
  --skip-existing-complete


In [ ]:
# Extract test: video only + emotion prompt, retain 30% of visual tokens.
!python {CODE}/scripts/run_qwen_shared_chunks.py \
  --index-csv {INDEX_ROOT}/meld_test_index.csv \
  --chunks-dir {CHUNKS_ROOT}/test \
  --chunk-prefix test \
  --chunk-size 500 \
  --fps 6 \
  --max-frames 64 \
  --min-frames 4 \
  --frame-size 224 \
  --pooling last \
  --prompt-style emotion_task \
  --modality-mode video_only \
  --visual-token-pruning divprune \
  --divprune-retain-ratio {RETAIN_RATIO} \
  --save-dtype float32 \
  --gc-every 5 \
  --skip-existing-complete


In [ ]:
# Merge dev/test chunks.
for split in ["dev", "test"]:
    !python {CODE}/scripts/merge_embedding_chunks.py \
      --chunks-dir {CHUNKS_ROOT}/{split} \
      --output-pt {EMBEDDINGS_ROOT}/meld_{split}.pt \
      --pattern "{split}_*.pt"


In [ ]:
# Verify the extraction condition and embedding integrity before evaluation.
SAMPLE_COUNTS = {}
for split in ["dev", "test"]:
    payload = torch.load(EMBEDDINGS_ROOT / f"meld_{split}.pt", map_location="cpu", weights_only=False)
    x = payload["embeddings"].float()
    assert x.ndim == 2 and x.shape[1] == 2048 and x.shape[0] > 0
    assert torch.isfinite(x).all(), f"Non-finite embeddings in {split}"
    assert x.shape[0] == len(payload["labels"]) == len(payload["sample_ids"])
    assert len(set(payload["sample_ids"])) == x.shape[0]
    assert payload["label_names"] == LABELS
    for item in payload["config"]["merged_from"]:
        config = item["config"]
        assert config["modality_mode"] == "video_only"
        assert config["prompt_style"] == "emotion_task"
        assert config["visual_token_pruning"] == "divprune"
        assert abs(config["divprune_retain_ratio"] - RETAIN_RATIO) < 1e-9
    SAMPLE_COUNTS[split] = len(payload["labels"])
    print(split, "shape:", tuple(x.shape), "extraction errors:", len(payload.get("errors", [])))
del payload, x


## Evaluate the existing full-baseline MLP

In [ ]:
for split in ["dev", "test"]:
    !python {CODE}/scripts/evaluate_mlp.py \
      --checkpoint {BASELINE_CHECKPOINT} \
      --embeddings-pt {EMBEDDINGS_ROOT}/meld_{split}.pt \
      --output-dir {BASELINE_RESULTS} \
      --split-name {split} \
      --device cuda


## Evaluate the existing 30%-DivPrune MLP

In [ ]:
for split in ["dev", "test"]:
    !python {CODE}/scripts/evaluate_mlp.py \
      --checkpoint {DIVPRUNE_CHECKPOINT} \
      --embeddings-pt {EMBEDDINGS_ROOT}/meld_{split}.pt \
      --output-dir {DIVPRUNE_RESULTS} \
      --split-name {split} \
      --device cuda


## Final performance comparison

Both MLPs are evaluated on identical embeddings and sample IDs. These results measure performance on the new video-only pruned representation. Align sample IDs before comparing with older runs if extraction failures differ.

In [ ]:
rows = []
METRICS = {}
for name, directory in [("Full-baseline MLP", BASELINE_RESULTS), ("30% DivPrune MLP", DIVPRUNE_RESULTS)]:
    for split in ["dev", "test"]:
        values = json.loads((directory / f"{split}_metrics.json").read_text())
        METRICS[(name, split)] = values
        rows.append({
            "Representation": "Video only + prompt, 30% DivPrune",
            "MLP": name, "Split": split, "Samples": SAMPLE_COUNTS[split],
            "Accuracy": values["accuracy"], "Macro F1": values["macro_f1"],
            "Weighted F1": values["weighted_f1"],
        })
results = pd.DataFrame(rows)
display(results.round(4))
results.to_csv(RESULTS_ROOT / "metrics_summary.csv", index=False)

for split in ["dev", "test"]:
    a = pd.read_csv(BASELINE_RESULTS / f"{split}_predictions.csv")
    b = pd.read_csv(DIVPRUNE_RESULTS / f"{split}_predictions.csv")
    assert a["sample_id"].equals(b["sample_id"])
    assert a["true_id"].equals(b["true_id"])


In [ ]:
# Test confusion matrices and per-class reports.
for name in ["Full-baseline MLP", "30% DivPrune MLP"]:
    values = METRICS[(name, "test")]
    print(name, "— test confusion matrix")
    display(pd.DataFrame(values["confusion_matrix"],
        index=[f"true_{label}" for label in LABELS],
        columns=[f"pred_{label}" for label in LABELS]))
    display(pd.DataFrame(values["classification_report"]).T.round(4))


In [ ]:
# Bundle compact results for download; embeddings/chunks remain in Kaggle output.
import shutil
from IPython.display import FileLink

archive_path = shutil.make_archive(
    "/kaggle/working/video_only_divprune_030_results", "zip", root_dir=RESULTS_ROOT)
print("Results:", RESULTS_ROOT)
print("Embeddings:", EMBEDDINGS_ROOT)
print("Chunks:", CHUNKS_ROOT)
display(FileLink(archive_path))
